In [0]:
%run /Workspace/Users/anupatil172006@gmail.com/Smart-Patient-Readmission-Risk-Pipeline/config/config.py

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

print("Silver transformation started.")
print(f"Database: {DATABASE}")
print(f"Silver table: {SILVER_ADMISSIONS}")

Silver transformation started.
Database: dbacademy.smart_readmission
Silver table: dbacademy.smart_readmission.silver_admissions_enriched


### Loading the bronze table

In [0]:
# ==========================================================
# Read Bronze Tables
# ==========================================================

patients_bronze = spark.table(BRONZE_PATIENTS)
diagnoses_bronze = spark.table(BRONZE_DIAGNOSES)
admissions_bronze = spark.table(BRONZE_ADMISSIONS)

print("Bronze tables loaded successfully.")

print(f"Patients:   {patients_bronze.count()}")
print(f"Diagnoses:  {diagnoses_bronze.count()}")
print(f"Admissions: {admissions_bronze.count()}")

Bronze tables loaded successfully.
Patients:   205
Diagnoses:  12
Admissions: 692


### Inspecting the schemas

In [0]:
# ==========================================================
# Inspect Bronze Schemas
# ==========================================================

print("========== PATIENTS BRONZE ==========")
patients_bronze.printSchema()

print("\n========== DIAGNOSES BRONZE ==========")
diagnoses_bronze.printSchema()

print("\n========== ADMISSIONS BRONZE ==========")
admissions_bronze.printSchema()

========== PATIENTS BRONZE ==========
root
 |-- patient_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- contact: string (nullable = true)


========== DIAGNOSES BRONZE ==========
root
 |-- diagnosis_id: string (nullable = true)
 |-- icd_code: string (nullable = true)
 |-- category: string (nullable = true)


========== ADMISSIONS BRONZE ==========
root
 |-- admission_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- diagnosis_id: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- department: string (nullable = true)
 |-- physician: string (nullable = true)
 |-- length_of_stay: integer (nullable = true)
 |-- readmitted_within_30_days: integer (nullable = true)



### Cleaning and standardizing the bronze data

In [0]:
# ==========================================================
# Silver - Data Cleaning & Standardization
# ==========================================================

patients_clean = (
    patients_bronze
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("gender", F.upper(F.trim(F.col("gender"))))
    .withColumn(
        "age",
        F.when(
            (F.col("age") >= 0) & (F.col("age") <= 120),
            F.col("age")
        ).otherwise(None)
    )
)

diagnoses_clean = (
    diagnoses_bronze
    .withColumn("diagnosis_id", F.trim(F.col("diagnosis_id")))
    .withColumn("icd_code", F.upper(F.trim(F.col("icd_code"))))
    .withColumn("category", F.trim(F.col("category")))
)

admissions_clean = (
    admissions_bronze
    .withColumn("admission_id", F.trim(F.col("admission_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("diagnosis_id", F.trim(F.col("diagnosis_id")))
    .withColumn("department", F.trim(F.col("department")))
    .withColumn("physician", F.trim(F.col("physician")))
    .withColumn(
        "length_of_stay",
        F.when(
            F.col("length_of_stay") > 0,
            F.col("length_of_stay")
        ).otherwise(None)
    )
    .withColumn(
        "readmitted_within_30_days",
        F.when(
            F.col("readmitted_within_30_days").isin(0, 1),
            F.col("readmitted_within_30_days")
        ).otherwise(None)
    )
)

print("✅ Bronze data cleaned and standardized.")

✅ Bronze data cleaned and standardized.


### Join Patient + Diagnosis + Admission

In [0]:
# ==========================================================
# Silver - Join Bronze Datasets
# ==========================================================

silver_df = (
    admissions_clean.alias("a")
    .join(
        patients_clean.alias("p"),
        F.col("a.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .join(
        diagnoses_clean.alias("d"),
        F.col("a.diagnosis_id") == F.col("d.diagnosis_id"),
        "left"
    )
    .select(
        F.col("a.admission_id"),
        F.col("a.patient_id"),
        F.col("p.name").alias("patient_name"),
        F.col("p.age"),
        F.col("p.gender"),

        F.col("a.diagnosis_id"),
        F.col("d.icd_code"),
        F.col("d.category").alias("diagnosis_category"),

        F.col("a.admission_date"),
        F.col("a.discharge_date"),
        F.col("a.department"),
        F.col("a.physician"),
        F.col("a.length_of_stay"),
        F.col("a.readmitted_within_30_days")
    )
)

print(f"Silver dataset rows: {silver_df.count()}")
print(f"Silver dataset columns: {len(silver_df.columns)}")

silver_df.show(10, truncate=False)

Silver dataset rows: 692
Silver dataset columns: 14
+------------+----------+---------------+---+------+------------+--------+------------------+--------------+--------------+----------------+---------+--------------+-------------------------+
|admission_id|patient_id|patient_name   |age|gender|diagnosis_id|icd_code|diagnosis_category|admission_date|discharge_date|department      |physician|length_of_stay|readmitted_within_30_days|
+------------+----------+---------------+---+------+------------+--------+------------------+--------------+--------------+----------------+---------+--------------+-------------------------+
|A000001     |P00009    |Arunima Dugal  |53 |M     |D002        |I50     |Cardiovascular    |2025-11-24    |2025-11-28    |General Medicine|Dr. Mehta|4             |0                        |
|A000002     |P00044    |Isaac Patil    |38 |M     |D007        |E11     |Endocrine         |2025-09-08    |2025-09-10    |General Medicine|Dr. Kumar|2             |0              

### Creating age group

In [0]:
# ==========================================================
# Silver - Age Group Feature
# ==========================================================

silver_df = (
    silver_df
    .withColumn(
        "age_group",
        F.when(F.col("age") < 18, "Pediatric")
        .when(F.col("age") < 36, "Young Adult")
        .when(F.col("age") < 51, "Middle Age")
        .when(F.col("age") < 66, "Senior")
        .otherwise("Elderly")
    )
)

print("Age group feature created successfully.")

silver_df.select(
    "patient_id",
    "age",
    "age_group"
).show(20, truncate=False)

Age group feature created successfully.
+----------+---+-----------+
|patient_id|age|age_group  |
+----------+---+-----------+
|P00009    |53 |Senior     |
|P00044    |38 |Middle Age |
|P00133    |24 |Young Adult|
|P00134    |45 |Middle Age |
|P00012    |43 |Middle Age |
|P00155    |77 |Elderly    |
|P00109    |59 |Senior     |
|P00103    |40 |Middle Age |
|P00053    |83 |Elderly    |
|P00169    |69 |Elderly    |
|P00081    |73 |Elderly    |
|P00137    |80 |Elderly    |
|P00093    |44 |Middle Age |
|P00097    |83 |Elderly    |
|P00079    |25 |Young Adult|
|P00073    |93 |Elderly    |
|P00193    |32 |Young Adult|
|P00104    |43 |Middle Age |
|P00040    |51 |Senior     |
|P00186    |39 |Middle Age |
+----------+---+-----------+
only showing top 20 rows


### Prionr admission count

In [0]:
# ==========================================================
# Silver - Prior Admission Count
# ==========================================================

patient_window = (
    Window
    .partitionBy("patient_id")
    .orderBy("admission_date", "admission_id")
)

silver_df = (
    silver_df
    .withColumn(
        "prior_admission_count",
        F.count("*").over(
            patient_window.rowsBetween(
                Window.unboundedPreceding,
                -1
            )
        )
    )
)

print("Prior admission count calculated successfully.")

silver_df.select(
    "patient_id",
    "admission_id",
    "admission_date",
    "prior_admission_count"
).orderBy(
    "patient_id",
    "admission_date"
).show(20, truncate=False)

Prior admission count calculated successfully.
+----------+------------+--------------+---------------------+
|patient_id|admission_id|admission_date|prior_admission_count|
+----------+------------+--------------+---------------------+
|P00001    |A000486     |2026-01-19    |0                    |
|P00001    |A000253     |2026-02-12    |1                    |
|P00002    |A000040     |2025-08-12    |0                    |
|P00002    |A000427     |2026-04-30    |1                    |
|P00002    |A000035     |2026-05-07    |2                    |
|P00002    |A000599     |2026-08-01    |3                    |
|P00003    |A000236     |2025-12-11    |0                    |
|P00003    |A000197     |2025-12-15    |1                    |
|P00003    |A000336     |2026-02-08    |2                    |
|P00003    |A000086     |2026-02-18    |3                    |
|P00003    |A000249     |2026-05-27    |4                    |
|P00003    |A000025     |2026-06-02    |5                    |
|P00004 

### Admission time Features

In [0]:
# ==========================================================
# Silver - Admission Time Features
# ==========================================================

silver_df = (
    silver_df
    .withColumn(
        "admission_month",
        F.date_format("admission_date", "yyyy-MM")
    )
    .withColumn(
        "admission_year",
        F.year("admission_date")
    )
)

print("Admission time features created successfully.")

silver_df.select(
    "admission_date",
    "admission_month",
    "admission_year"
).show(20, truncate=False)

Admission time features created successfully.
+--------------+---------------+--------------+
|admission_date|admission_month|admission_year|
+--------------+---------------+--------------+
|2025-11-24    |2025-11        |2025          |
|2025-09-08    |2025-09        |2025          |
|2025-12-26    |2025-12        |2025          |
|2025-11-22    |2025-11        |2025          |
|2026-05-30    |2026-05        |2026          |
|2026-02-08    |2026-02        |2026          |
|2026-03-30    |2026-03        |2026          |
|2026-04-02    |2026-04        |2026          |
|2026-03-26    |2026-03        |2026          |
|2025-10-05    |2025-10        |2025          |
|2025-10-05    |2025-10        |2025          |
|2026-01-14    |2026-01        |2026          |
|2026-05-28    |2026-05        |2026          |
|2026-06-18    |2026-06        |2026          |
|2025-12-03    |2025-12        |2025          |
|2026-06-21    |2026-06        |2026          |
|2025-08-28    |2025-08        |2025      

### Comorbidity Proxy

In [0]:
# ==========================================================
# Silver - Comorbidity Proxy
# ==========================================================

patient_diagnosis_counts = (
    silver_df
    .groupBy("patient_id")
    .agg(
        F.countDistinct("diagnosis_category")
        .alias("distinct_diagnosis_categories")
    )
)

silver_df = (
    silver_df
    .join(
        patient_diagnosis_counts,
        on="patient_id",
        how="left"
    )
    .withColumn(
        "comorbidity_index",
        F.when(
            F.col("distinct_diagnosis_categories") >= 3,
            "High"
        )
        .when(
            F.col("distinct_diagnosis_categories") == 2,
            "Moderate"
        )
        .otherwise("Low")
    )
)

print("Comorbidity proxy created successfully.")

silver_df.select(
    "patient_id",
    "distinct_diagnosis_categories",
    "comorbidity_index"
).show(20, truncate=False)

Comorbidity proxy created successfully.
+----------+-----------------------------+-----------------+
|patient_id|distinct_diagnosis_categories|comorbidity_index|
+----------+-----------------------------+-----------------+
|P00137    |6                            |High             |
|P00040    |4                            |High             |
|P00134    |4                            |High             |
|P00103    |4                            |High             |
|P00169    |5                            |High             |
|P00073    |1                            |Low              |
|P00133    |5                            |High             |
|P00109    |5                            |High             |
|P00104    |2                            |Moderate         |
|P00012    |4                            |High             |
|P00093    |3                            |High             |
|P00079    |5                            |High             |
|P00193    |3                            |Hig

### Final Silver Dataset

In [0]:
# ==========================================================
# Silver - Final Dataset
# ==========================================================

silver_df = silver_df.select(
    "admission_id",
    "patient_id",
    "patient_name",
    "age",
    "age_group",
    "gender",
    "diagnosis_id",
    "icd_code",
    "diagnosis_category",
    "department",
    "physician",
    "admission_date",
    "discharge_date",
    "admission_month",
    "admission_year",
    "length_of_stay",
    "prior_admission_count",
    "distinct_diagnosis_categories",
    "comorbidity_index",
    "readmitted_within_30_days"
)

print(f"Silver rows: {silver_df.count()}")
print(f"Silver columns: {len(silver_df.columns)}")

silver_df.printSchema()
silver_df.show(10, truncate=False)

Silver rows: 692
Silver columns: 20
root
 |-- admission_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_group: string (nullable = false)
 |-- gender: string (nullable = true)
 |-- diagnosis_id: string (nullable = true)
 |-- icd_code: string (nullable = true)
 |-- diagnosis_category: string (nullable = true)
 |-- department: string (nullable = true)
 |-- physician: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- admission_month: string (nullable = true)
 |-- admission_year: integer (nullable = true)
 |-- length_of_stay: integer (nullable = true)
 |-- prior_admission_count: long (nullable = false)
 |-- distinct_diagnosis_categories: long (nullable = true)
 |-- comorbidity_index: string (nullable = false)
 |-- readmitted_within_30_days: integer (nullable = true)

+------------+----------+-------------+---+----

### Silver Data Quanlity Validation

In [0]:
# ==========================================================
# Silver - Data Quality Validation
# ==========================================================

print("========== SILVER VALIDATION ==========")

print(f"Total rows: {silver_df.count()}")

print(
    f"Distinct admissions: "
    f"{silver_df.select('admission_id').distinct().count()}"
)

print(
    f"Missing patient names: "
    f"{silver_df.filter(F.col('patient_name').isNull()).count()}"
)

print(
    f"Missing diagnosis categories: "
    f"{silver_df.filter(F.col('diagnosis_category').isNull()).count()}"
)

print(
    f"Invalid LOS: "
    f"{silver_df.filter(F.col('length_of_stay') <= 0).count()}"
)

print(
    f"Invalid readmission flags: "
    f"{silver_df.filter(
        ~F.col('readmitted_within_30_days').isin(0, 1)
    ).count()}"
)

========== SILVER VALIDATION ==========
Total rows: 692
Distinct admissions: 692
Missing patient names: 0
Missing diagnosis categories: 0
Invalid LOS: 0
Invalid readmission flags: 0


### Write Silver Delta Table

In [0]:
# ==========================================================
# Write Silver Delta Table
# ==========================================================

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_ADMISSIONS)
)

print("✅ Silver table written successfully.")
print(f"Table: {SILVER_ADMISSIONS}")

✅ Silver table written successfully.
Table: dbacademy.smart_readmission.silver_admissions_enriched


### Final Silver Check

In [0]:
# ==========================================================
# Silver - Final Verification
# ==========================================================

silver_check = spark.table(SILVER_ADMISSIONS)

print(f"Silver table rows: {silver_check.count()}")
print(f"Silver table columns: {len(silver_check.columns)}")

silver_check.show(10, truncate=False)

Silver table rows: 692
Silver table columns: 20
+------------+----------+-------------+---+----------+------+------------+--------+------------------+----------------+----------+--------------+--------------+---------------+--------------+--------------+---------------------+-----------------------------+-----------------+-------------------------+
|admission_id|patient_id|patient_name |age|age_group |gender|diagnosis_id|icd_code|diagnosis_category|department      |physician |admission_date|discharge_date|admission_month|admission_year|length_of_stay|prior_admission_count|distinct_diagnosis_categories|comorbidity_index|readmitted_within_30_days|
+------------+----------+-------------+---+----------+------+------------+--------+------------------+----------------+----------+--------------+--------------+---------------+--------------+--------------+---------------------+-----------------------------+-----------------+-------------------------+
|A000486     |P00001    |Aryan Maharaj|61 |